In [1]:
import pandas as pd
import numpy as np

# =========================
# 0. 데이터 로드
# =========================

import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'  # 한글 폰트
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 깨짐 방지

df = pd.read_csv(r"C:\Users\Admin\Desktop\hhh\Horse\data_preprocessing\merged_data_kr_Nan.csv", encoding="utf-8-sig")


# =========================
# 1. 기본 전처리
# =========================

# 날짜 변환
df['경주일자'] = pd.to_datetime(df['경주일자'], errors='coerce')

# 출생일 정리 (YYYYMMDD)
df['출생일'] = df['출생일'].astype(str).str.extract(r'(\d{8})')
df['출생일'] = pd.to_datetime(df['출생일'], errors='coerce')

# 나이
df['나이'] = (df['경주일자'] - df['출생일']).dt.days / 365

# =========================
# 2. 정렬 (중요)
# =========================
df = df.sort_values(['마명', '경주일자'])

# =========================
# 3. 말 단위 rolling feature
# =========================

# 최근 순위 기반 (target proxy)
for w in [3, 5, 10]:
    df[f'마명_최근{w}평균순위'] = (
        df.groupby('마명')['출주두수_대비_상대순위점수']
        .transform(lambda x: x.shift(1).rolling(w).mean())
    )

    df[f'마명_최근{w}표준편차'] = (
        df.groupby('마명')['출주두수_대비_상대순위점수']
        .transform(lambda x: x.shift(1).rolling(w).std())
    )

# slope (추세)
def slope(x):
    if len(x) < 3:
        return np.nan
    y = np.array(x)
    return np.polyfit(range(len(y)), y, 1)[0]


df['마명_최근3slope'] = (
    df.groupby('마명')['출주두수_대비_상대순위점수']
    .transform(lambda x: x.shift(1).rolling(3).apply(slope, raw=False))
)

# =========================
# 4. 기수 / 조교사 능력
# =========================

for w in [5, 10]:
    df[f'기수_최근{w}평균'] = (
        df.groupby('기수번호')['출주두수_대비_상대순위점수']
        .transform(lambda x: x.shift(1).rolling(w).mean())
    )

    df[f'조교사_최근{w}평균'] = (
        df.groupby('조교사번호')['출주두수_대비_상대순위점수']
        .transform(lambda x: x.shift(1).rolling(w).mean())
    )

# =========================
# 5. 말 × 기수 interaction
# =========================

df['마명_기수'] = df['마명'].astype(str) + '_' + df['기수번호'].astype(str)

for w in [10]:
    df[f'마명_기수_승률'] = (
        df.groupby('마명_기수')['출주두수_대비_상대순위점수']
        .transform(lambda x: x.shift(1).rolling(w).mean())
    )

# =========================
# 6. 체중 변화
# =========================

df['마체중_변화'] = df.groupby('마명')['마체중'].diff()
df['마체중_롤링평균'] = (
    df.groupby('마명')['마체중'].transform(lambda x: x.shift(1).rolling(3).mean())
)

# =========================
# 7. 휴식일 (핵심)
# =========================

df['이전경주일'] = df.groupby('마명')['경주일자'].shift(1)
df['휴식일'] = (df['경주일자'] - df['이전경주일']).dt.days

# =========================
# 8. 경주 강도 (race strength)
# =========================

race_group = df.groupby('경주번호')

df['경주_평균능력'] = race_group['출주두수_대비_상대순위점수'].transform('mean')
df['경주_표준편차'] = race_group['출주두수_대비_상대순위점수'].transform('std')
df['경주_상위3평균'] = race_group['출주두수_대비_상대순위점수'].transform(
    lambda x: x.nlargest(3).mean()
)

# =========================
# 9. 상대 위치 (ranking within race)
# =========================

df['경주_퍼센타일'] = df.groupby('경주번호')['출주두수_대비_상대순위점수'].rank(pct=True)

# =========================
# 10. 거리 / 등급 encoding
# =========================

df['경주등급_enc'] = df['경주등급'].astype('category').cat.codes

df['경주로상태_enc'] = df['경주로상태'].astype('category').cat.codes

df['날씨_enc'] = df['날씨'].astype('category').cat.codes

# =========================
# 11. 나이 비선형
# =========================

df['나이2'] = df['나이'] ** 2

# =========================
# 12. 최종 정리
# =========================

drop_cols = ['이전경주일']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# 결측 처리 (LightGBM용)
df = df.replace([np.inf, -np.inf], np.nan)

print("Feature Engineering Complete")
print(df.shape)


Feature Engineering Complete
(15460, 50)


In [2]:
df.head()

,분할경주여부,마명,마번,기수번호,조교사번호,부담구분,출전번호,경주일자,경주등급,출전마구분,...,마체중_롤링평균,휴식일,경주_평균능력,경주_표준편차,경주_상위3평균,경주_퍼센타일,경주등급_enc,경주로상태_enc,날씨_enc,나이2
3546,0,가야스타,51451,080530,70165.0,2,1,2024-09-08,일반,루키,...,NaN,NaN,0.544266,0.313014,1.0,0.557143,2,1,2,NaN
3877,0,가야스타,51451,080550,70165.0,2,10,2024-10-12,일반,일반,...,NaN,34.0,0.567665,0.312228,1.0,0.195605,2,0,2,NaN
4374,0,가야스타,51451,080550,70165.0,2,10,2024-11-17,일반,루키,...,NaN,36.0,0.544266,0.313014,1.0,0.346032,2,3,4,NaN
4907,1,가야스타,51451,080530,70165.0,2,2,2024-12-22,일반,일반,...,441.666667,35.0,0.564287,0.311149,1.0,0.247079,2,3,2,NaN
5646,0,가야스타,51451,080530,70165.0,2,7,2025-02-09,일반,일반,...,446.000000,49.0,0.564287,0.311149,1.0,0.166667,2,0,2,NaN


In [3]:
df.isnull().sum()

분할경주여부                0
마명                    0
마번                    0
기수번호                  0
조교사번호                 0
부담구분                  0
출전번호                  0
경주일자                  0
경주등급                  0
출전마구분                 0
경주번호                  0
야간경마여부                0
마필등급                  0
출주두수                  0
경주로상태                 0
날씨                    0
마체중                   0
출생일               15460
성별                    0
소유자명                  0
생산국                   0
부마명                   0
소재지                   0
출주두수_대비_상대순위점수        0
평균_시속                 0
나이                15460
마명_최근3평균순위         3811
마명_최근3표준편차         3811
마명_최근5평균순위         5907
마명_최근5표준편차         5907
마명_최근10평균순위        9789
마명_최근10표준편차        9789
마명_최근3slope        3811
기수_최근5평균            505
조교사_최근5평균           245
기수_최근10평균           971
조교사_최근10평균          486
마명_기수                 0
마명_기수_승률          15249
마체중_변화             1341
마체중_롤링평균           3811
휴식일             

In [4]:
df.info()

<class 'pandas.DataFrame'>
Index: 15460 entries, 3546 to 15365
Data columns (total 50 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   분할경주여부          15460 non-null  int64         
 1   마명              15460 non-null  str           
 2   마번              15460 non-null  int64         
 3   기수번호            15460 non-null  str           
 4   조교사번호           15460 non-null  float64       
 5   부담구분            15460 non-null  int64         
 6   출전번호            15460 non-null  int64         
 7   경주일자            15460 non-null  datetime64[us]
 8   경주등급            15460 non-null  str           
 9   출전마구분           15460 non-null  str           
 10  경주번호            15460 non-null  int64         
 11  야간경마여부          15460 non-null  str           
 12  마필등급            15460 non-null  float64       
 13  출주두수            15460 non-null  float64       
 14  경주로상태           15460 non-null  str           
 15  날씨             